In [1]:
!pip install fastdtw pandas numpy scipy scikit-learn catboost


In [2]:
import pandas as pd
import numpy as np
import time
import warnings
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score, recall_score
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
print("Libraries imported successfully!")


Libraries imported successfully!


## 1. TIỀN XỬ LÝ (LỌC NHIỄU, TÁI TẠO, VÀ CHUẨN HÓA 2D)

- **is_dancing:** Lọc video học viên đứng yên dựa trên độ lệch chuẩn (std) tọa độ các khớp cổ tay và cổ chân (ngưỡng `variance_threshold = 0.020` để loại bỏ các video nhiễu tĩnh như Video 6 có `std = 0.0193`).
- **preprocess_data:** 
  1. Xử lý mất dấu (Visibility < 0.5) thành NaN và bù đắp bằng nội suy tuyến tính (`linear interpolation`).
  2. Dịch gốc tọa độ về trung điểm của hai khớp hông (`left_hip` và `right_hip`).
  3. Chuẩn hóa tỷ lệ nhân trắc học theo độ dài thân người (`torso_length` đo giữa trung điểm hai vai và trung điểm hai hông trên mặt phẳng 2D để loại bỏ nhiễu trục Z).


In [3]:
# [NOISE ISOLATION] - Kiểm tra xem học viên có đang chuyển động hay đứng yên
def is_dancing(df_student, variance_threshold=0.020):
    key_joints = ['left_wrist', 'right_wrist', 'left_ankle', 'right_ankle']
    df_key = df_student[df_student['Joint_Name'].isin(key_joints)]
    movement_std = df_key.groupby('Joint_Name')[['X', 'Y']].std().mean().mean()
    return movement_std >= variance_threshold, movement_std

# [SIGNAL RECONSTRUCTION & ANATOMICAL SCALING]
def preprocess_data(df):
    df = df.sort_values(by=['Frame', 'Joint_Name']).reset_index(drop=True)
    
    # 1. Mất dấu -> NaN -> nội suy tuyến tính
    mask = df['Visibility'] < 0.5
    df.loc[mask, ['X', 'Y', 'Z']] = np.nan
    df[['X', 'Y', 'Z']] = df.groupby('Joint_Name')[['X', 'Y', 'Z']].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both')
    )
    df.fillna(0, inplace=True)
    
    normalized_frames = []
    for frame_id, frame_data in df.groupby('Frame'):
        l_hip = frame_data[frame_data['Joint_Name'] == 'left_hip']
        r_hip = frame_data[frame_data['Joint_Name'] == 'right_hip']
        l_sh = frame_data[frame_data['Joint_Name'] == 'left_shoulder']
        r_sh = frame_data[frame_data['Joint_Name'] == 'right_shoulder']
        
        if not l_hip.empty and not r_hip.empty and not l_sh.empty and not r_sh.empty:
            root_x = (l_hip['X'].values[0] + r_hip['X'].values[0]) / 2
            root_y = (l_hip['Y'].values[0] + r_hip['Y'].values[0]) / 2
            
            sh_x = (l_sh['X'].values[0] + r_sh['X'].values[0]) / 2
            sh_y = (l_sh['Y'].values[0] + r_sh['Y'].values[0]) / 2
            
            # Chỉ dùng 2D (X, Y) để đo Torso, loại bỏ nhiễu trục Z
            torso_length = np.sqrt((sh_x - root_x)**2 + (sh_y - root_y)**2)
            torso_length = max(torso_length, 1e-5)
        else:
            root_x, root_y = 0, 0
            torso_length = 1.0
            
        frame_data = frame_data.copy()
        frame_data['X_norm'] = (frame_data['X'] - root_x) / torso_length
        frame_data['Y_norm'] = (frame_data['Y'] - root_y) / torso_length
        frame_data['Z_norm'] = frame_data['Z']
        
        normalized_frames.append(frame_data)
        
    return pd.concat(normalized_frames, ignore_index=True)


## 2. XỬ LÝ ĐẶC TRƯNG (KHOẢNG CÁCH, GÓC ĐỘ VÀ VẬN TỐC 2D)

Sử dụng thuật toán **FastDTW** để đồng bộ hóa chuỗi thời gian giữa video học viên và video giáo viên. Sau khi đồng bộ, trích xuất 3 đặc trưng cốt lõi cho từng khớp tại mỗi khung hình:
1. **Đặc trưng Không gian (`Joint_Error`):** Sai số khoảng cách Euclidean giữa khớp học viên và giáo viên.
2. **Đặc trưng Hướng (`Joint_Cosine`):** Độ tương đồng Cosine đo góc lệch vector hướng giữa học viên và giáo viên.
3. **Đặc trưng Động lực học (`Velocity_Error`):** Độ lệch vector vận tốc (đạo hàm bậc nhất theo thời gian của tọa độ 2D giữa hai khung hình liên tiếp).


In [4]:
def extract_joint_features(df_student, df_teacher):
    def get_vectors(df):
        wide = df.pivot(index='Frame', columns='Joint_Name', values=['X_norm', 'Y_norm']).fillna(0)
        return wide.values, wide.index.tolist()

    vec_s, frames_s = get_vectors(df_student)
    vec_t, frames_t = get_vectors(df_teacher)
    
    # Đồng bộ hóa chuỗi thời gian bằng FastDTW
    distance, path = fastdtw(vec_s, vec_t, dist=euclidean)
    aligned_pairs = [{'Frame_s': frames_s[s], 'Frame_t': frames_t[t]} for s, t in path]
    df_path = pd.DataFrame(aligned_pairs)
    
    df_s_aligned = pd.merge(df_path, df_student, left_on='Frame_s', right_on='Frame').drop(columns=['Frame'])
    df_t_aligned = pd.merge(df_path, df_teacher, left_on='Frame_t', right_on='Frame').drop(columns=['Frame'])
    
    df_s_aligned = df_s_aligned.rename(columns={'X_norm': 'X_s', 'Y_norm': 'Y_s'})
    df_t_aligned = df_t_aligned.rename(columns={'X_norm': 'X_t', 'Y_norm': 'Y_t'})
    
    label_col = 'Label' if 'Label' in df_s_aligned.columns else ('Label_s' if 'Label_s' in df_s_aligned.columns else None)
    cols_to_merge_s = ['Frame_s', 'Frame_t', 'Joint_Name', 'X_s', 'Y_s'] + ([label_col] if label_col else [])
    
    df_merged = pd.merge(df_s_aligned[cols_to_merge_s], 
                         df_t_aligned[['Frame_s', 'Frame_t', 'Joint_Name', 'X_t', 'Y_t']], 
                         on=['Frame_s', 'Frame_t', 'Joint_Name'])
    
    # 1. Đặc trưng Không gian: Sai số khoảng cách Euclidean (Joint_Error)
    df_merged['Joint_Error'] = np.sqrt((df_merged['X_s'] - df_merged['X_t'])**2 + 
                                       (df_merged['Y_s'] - df_merged['Y_t'])**2)
                                       
    # 2. Đặc trưng Hướng: Độ tương đồng Cosine (Joint_Cosine)
    dot = (df_merged['X_s']*df_merged['X_t']) + (df_merged['Y_s']*df_merged['Y_t'])
    norm_s = np.sqrt(df_merged['X_s']**2 + df_merged['Y_s']**2)
    norm_t = np.sqrt(df_merged['X_t']**2 + df_merged['Y_t']**2)
    df_merged['Joint_Cosine'] = dot / (norm_s * norm_t + 1e-9)
    
    # 3. Đặc trưng Động lực học: Sai số Vận tốc (Velocity_Error)
    df_merged['Vel_X_s'] = df_merged.groupby('Joint_Name')['X_s'].diff().fillna(0)
    df_merged['Vel_Y_s'] = df_merged.groupby('Joint_Name')['Y_s'].diff().fillna(0)
    df_merged['Vel_X_t'] = df_merged.groupby('Joint_Name')['X_t'].diff().fillna(0)
    df_merged['Vel_Y_t'] = df_merged.groupby('Joint_Name')['Y_t'].diff().fillna(0)
    
    df_merged['Velocity_Error'] = np.sqrt((df_merged['Vel_X_s'] - df_merged['Vel_X_t'])**2 + 
                                          (df_merged['Vel_Y_s'] - df_merged['Vel_Y_t'])**2)
    
    output_cols = ['Frame_s', 'Joint_Name', 'Joint_Error', 'Joint_Cosine', 'Velocity_Error']
    if label_col:
        df_merged.rename(columns={label_col: 'Ground_Truth_Label'}, inplace=True)
        output_cols.append('Ground_Truth_Label')
        
    return df_merged[output_cols]


## 3. TRÍCH XUẤT DATASET TỪ CÁC CẶP VIDEO

Xây dựng tập dữ liệu hợp nhất `master_dataset` từ 13 cặp video học viên - giáo viên thực tế (bỏ cặp 13 là giáo viên tự so sánh, cặp 14 không có file).
Dữ liệu trích xuất sẽ được lưu đệm (cache) để tái sử dụng nhanh chóng.


In [5]:
import os

dance_pairs = [
    {"id": 1, "student": "student_dance_1.csv", "teacher": "teacher_dance_1.csv"},
    {"id": 2, "student": "student_dance_2.csv", "teacher": "teacher_dance_2.csv"},
    {"id": 3, "student": "student_dance_3.csv", "teacher": "teacher_dance_3.csv"},
    {"id": 4, "student": "student_dance_4.csv", "teacher": "teacher_dance_4.csv"},
    {"id": 5, "student": "student_dance_5.csv", "teacher": "teacher_dance_5.csv"},
    {"id": 6, "student": "student_dance_6.csv", "teacher": "teacher_dance_6.csv"},
    {"id": 7, "student": "student_dance_7.csv", "teacher": "teacher_dance_7.csv"},
    {"id": 8, "student": "student_dance_8.csv", "teacher": "teacher_dance_8.csv"},
    {"id": 9, "student": "student_dance_9.csv", "teacher": "teacher_dance_9.csv"},
    {"id": 10, "student": "student_dance_10.csv", "teacher": "teacher_dance_10.csv"},
    {"id": 11, "student": "student_dance_11.csv", "teacher": "teacher_dance_11.csv"},
    {"id": 12, "student": "student_dance_12.csv", "teacher": "teacher_dance_12.csv"},
    {"id": 15, "student": "student_dance_15.csv", "teacher": "teacher_dance_15.csv"}
]

cache_file = "master_dataset_v020_cache.pkl"

if os.path.exists(cache_file):
    print(f"[*] Đang tải dữ liệu từ tệp đệm: {cache_file}...")
    master_dataset = pd.read_pickle(cache_file)
    print(f"[OK] Đã tải xong dataset: {len(master_dataset):,} hàng từ {master_dataset['Video_ID'].nunique()} video.")
else:
    all_joints_data = []
    print("=== BẮT ĐẦU TRÍCH XUẤT CẤP ĐỘ KHỚP ===")
    for pair in dance_pairs:
        try:
            df_s = pd.read_csv(pair["student"])
            df_t = pd.read_csv(pair["teacher"])
            
            is_moving, std = is_dancing(df_s)
            if not is_moving:
                print(f"[BỘ LỌC] Bỏ qua Video {pair['id']}: Đứng yên (Std={std:.4f})")
                continue
                
            print(f"[V] Đang xử lý Video {pair['id']}...")
            df_s_clean = preprocess_data(df_s)
            df_t_clean = preprocess_data(df_t)
            
            joint_features_df = extract_joint_features(df_s_clean, df_t_clean)
            joint_features_df['Video_ID'] = pair['id']
            all_joints_data.append(joint_features_df)
            
        except Exception as e:
            print(f"[-] Lỗi tại Video {pair['id']}: {e}")

    master_dataset = pd.concat(all_joints_data, ignore_index=True)
    master_dataset.to_pickle(cache_file)
    print(f"[OK] Hoàn tất và lưu tệp đệm: {cache_file}")


=== BẮT ĐẦU TRÍCH XUẤT CẤP ĐỘ KHỚP ===
[V] Đang xử lý Video 1...
[V] Đang xử lý Video 2...
[V] Đang xử lý Video 3...
[V] Đang xử lý Video 4...
[V] Đang xử lý Video 5...
[BỘ LỌC] Bỏ qua Video 6: Đứng yên (Std=0.0193)
[V] Đang xử lý Video 7...
[V] Đang xử lý Video 8...
[V] Đang xử lý Video 9...
[V] Đang xử lý Video 10...
[V] Đang xử lý Video 11...
[V] Đang xử lý Video 12...
[V] Đang xử lý Video 15...
[OK] Hoàn tất và lưu tệp đệm: master_dataset_v020_cache.pkl


## 4. PHÂN CHIA DỮ LIỆU ĐỘC LẬP TRAIN / VALIDATION / TEST (THEO VIDEO_ID)

> [!IMPORTANT]
> **Vấn đề ở bản cũ (`processing_11`):** Dò tìm ngưỡng tối ưu $T^*$ trực tiếp trên tập **Test**, dẫn đến rò rỉ dữ liệu (data leakage) và kết quả đánh giá thiên vị lạc quan.
>
> **Giải pháp chuẩn hóa trong bản `processing_11_ver1`:**
> 1. Bộ lọc nhiễu tĩnh (`is_dancing`, ngưỡng `0.020`) tự động loại bỏ Video 6 ngay từ Phase 1.
> 2. Phân chia 12 video hợp lệ còn lại theo **Video_ID** (Group-level split) để đảm bảo không rò rỉ khung hình giữa các tập:
>    - **Tập Test độc lập (`TEST_VIDEO_IDS = [5, 9]`):** Gồm 2 video (613 frames: 410 Pass, 203 Fail). Giữ nguyên cặp video này để làm đối chứng khách quan với bản cũ.
>    - **Tập Validation độc lập (`VAL_VIDEO_IDS = [3, 7]`):** Gồm 2 video (892 frames: 596 Pass, 296 Fail), đại diện cho cả động tác chuẩn xác và lỗi sai. Tập này **chỉ dùng để tối ưu hóa ngưỡng điểm $T^*$**.
>    - **Tập Train (`TRAIN_VIDEO_IDS`):** Gồm 8 video hợp lệ còn lại (`[1, 2, 4, 8, 10, 11, 12, 15]`) dùng để huấn luyện mô hình chấm khớp CatBoost.
> 3. Quy trình kiểm định độc lập:
>    $$\text{Train Model} \longrightarrow \text{Tìm } T^* \text{ trên Validation} \longrightarrow \text{Cố định } T^* \text{ và Đánh giá trên Test}$$


In [6]:
# Định nghĩa danh sách Video cho từng tập độc lập
TEST_VIDEO_IDS = [5, 9]     # Tập kiểm thử độc lập (chuẩn đối chứng)
VAL_VIDEO_IDS = [3, 7]      # Tập kiểm định (dùng tìm ngưỡng T*)
TRAIN_VIDEO_IDS = sorted([v for v in master_dataset['Video_ID'].unique() 
                          if v not in TEST_VIDEO_IDS and v not in VAL_VIDEO_IDS])

train_data = master_dataset[master_dataset['Video_ID'].isin(TRAIN_VIDEO_IDS)].reset_index(drop=True)
val_data = master_dataset[master_dataset['Video_ID'].isin(VAL_VIDEO_IDS)].reset_index(drop=True)
test_data = master_dataset[master_dataset['Video_ID'].isin(TEST_VIDEO_IDS)].reset_index(drop=True)

# Thống kê số lượng khung hình (Frame) và số dòng dữ liệu cấp khớp (Joint sample)
n_train_frames = train_data[['Video_ID', 'Frame_s']].drop_duplicates().shape[0]
n_val_frames = val_data[['Video_ID', 'Frame_s']].drop_duplicates().shape[0]
n_test_frames = test_data[['Video_ID', 'Frame_s']].drop_duplicates().shape[0]

print("==========================================================================")
print("                   THỐNG KÊ PHÂN CHIA DỮ LIỆU ĐỘC LẬP                     ")
print("==========================================================================")
print(f"1. TẬP HUẤN LUYỆN (TRAIN)     : {len(TRAIN_VIDEO_IDS)} videos {TRAIN_VIDEO_IDS}")
print(f"   -> Số lượng Frame          : {n_train_frames:,} frames ({len(train_data):,} mẫu khớp)")
print(f"   -> Tỉ lệ nhãn khớp (1/0)   : {(train_data['Ground_Truth_Label']==1).mean()*100:.1f}% Pass / {(train_data['Ground_Truth_Label']==0).mean()*100:.1f}% Fail\n")

print(f"2. TẬP KIỂM ĐỊNH (VALIDATION) : {len(VAL_VIDEO_IDS)} videos {VAL_VIDEO_IDS}")
print(f"   -> Số lượng Frame          : {n_val_frames:,} frames ({len(val_data):,} mẫu khớp)")
print(f"   -> Tỉ lệ nhãn khớp (1/0)   : {(val_data['Ground_Truth_Label']==1).mean()*100:.1f}% Pass / {(val_data['Ground_Truth_Label']==0).mean()*100:.1f}% Fail\n")

print(f"3. TẬP KIỂM THỬ (TEST)        : {len(TEST_VIDEO_IDS)} videos {TEST_VIDEO_IDS}")
print(f"   -> Số lượng Frame          : {n_test_frames:,} frames ({len(test_data):,} mẫu khớp)")
print(f"   -> Tỉ lệ nhãn khớp (1/0)   : {(test_data['Ground_Truth_Label']==1).mean()*100:.1f}% Pass / {(test_data['Ground_Truth_Label']==0).mean()*100:.1f}% Fail")
print("==========================================================================")


                   THỐNG KÊ PHÂN CHIA DỮ LIỆU ĐỘC LẬP                     
1. TẬP HUẤN LUYỆN (TRAIN)     : 8 videos [np.int64(1), np.int64(2), np.int64(4), np.int64(8), np.int64(10), np.int64(11), np.int64(12), np.int64(15)]
   -> Số lượng Frame          : 4,544 frames (60,190 mẫu khớp)
   -> Tỉ lệ nhãn khớp (1/0)   : 50.8% Pass / 49.2% Fail

2. TẬP KIỂM ĐỊNH (VALIDATION) : 2 videos [3, 7]
   -> Số lượng Frame          : 892 frames (12,090 mẫu khớp)
   -> Tỉ lệ nhãn khớp (1/0)   : 57.7% Pass / 42.3% Fail

3. TẬP KIỂM THỬ (TEST)        : 2 videos [5, 9]
   -> Số lượng Frame          : 613 frames (8,047 mẫu khớp)
   -> Tỉ lệ nhãn khớp (1/0)   : 64.3% Pass / 35.7% Fail


## 5. HUẤN LUYỆN MÔ HÌNH CATBOOST CHẤM KHỚP (CHỈ TRÊN TẬP TRAIN)

Mô hình CatBoost Classifier được huấn luyện trên 3 đặc trưng không gian - động lực học:
$$X = [\text{Joint\_Error}, \text{Joint\_Cosine}, \text{Velocity\_Error}]$$
Mô hình hoàn toàn **không tiếp cận dữ liệu của tập Validation và tập Test** trong quá trình fit.


In [7]:
feature_cols = ['Joint_Error', 'Joint_Cosine', 'Velocity_Error']

X_train = train_data[feature_cols]
y_train = train_data['Ground_Truth_Label']

X_val = val_data[feature_cols]
y_val = val_data['Ground_Truth_Label']

X_test = test_data[feature_cols]
y_test = test_data['Ground_Truth_Label']

print("Đang huấn luyện mô hình CatBoost Classifier trên tập TRAIN...")
cb_model = CatBoostClassifier(
    iterations=150,
    depth=6,
    learning_rate=0.08,
    random_seed=SEED,
    verbose=0
)
cb_model.fit(X_train, y_train)
print("[OK] Đã huấn luyện xong mô hình CatBoost!")

# Hiển thị mức độ quan trọng của từng đặc trưng
importances = cb_model.get_feature_importance()
print("\nĐộ quan trọng của các đặc trưng:")
for col, imp in zip(feature_cols, importances):
    print(f"  - {col:<15}: {imp:.2f}%")


Đang huấn luyện mô hình CatBoost Classifier trên tập TRAIN...
[OK] Đã huấn luyện xong mô hình CatBoost!

Độ quan trọng của các đặc trưng:
  - Joint_Error    : 50.67%
  - Joint_Cosine   : 27.27%
  - Velocity_Error : 22.06%


## 6. TỐI ƯU HÓA NGƯỠNG ĐIỂM $T^*$ TRÊN TẬP VALIDATION

1. Dùng mô hình CatBoost dự đoán nhãn cho 13 khớp trên tập **Validation**.
2. Với mỗi khung hình, tính tổng điểm $S_{\text{frame}}$ theo trọng số barem giải phẫu:
   $$S_{\text{frame}} = \sum_{j \in \text{Joints}} \hat{y}_j \times w_j, \quad S_{\text{frame}} \in [0, 13]$$
3. Thực hiện **Grid Search** quét ngưỡng $T \in [6.0, 9.0]$ trên tập Validation.
4. Tiêu chí chọn ngưỡng tối ưu $T^*$:
   - Đảm bảo độ nhạy phát hiện lỗi sai: $\text{Recall}_{\text{FAIL}} \ge 85\%$.
   - Tối đa hóa chỉ số cân bằng: $\text{Macro-F1}$.
   - Nếu không có ngưỡng nào thỏa mãn $\text{Recall}_{\text{FAIL}} \ge 85\%$, chọn ngưỡng có $\text{Macro-F1}$ cao nhất.


In [8]:
# Barem trọng số giải phẫu (13 khớp MediaPipe)
weights = {
    'left_elbow': 1.0, 'right_elbow': 1.0, 'left_knee': 1.0, 'right_knee': 1.0,
    'left_wrist': 1.0, 'right_wrist': 1.0, 'left_ankle': 1.0, 'right_ankle': 1.0,
    'left_shoulder': 1.0, 'right_shoulder': 1.0, 'left_hip': 1.0, 'right_hip': 1.0,
    'nose': 1.0
}

# Dự đoán nhãn khớp trên tập Validation
val_data['Predicted_Joint_Label'] = cb_model.predict(X_val)

# Tính tổng điểm cấp độ khung hình trên tập Validation
val_frame_records = []
for (vid, fid), frame_df in val_data.groupby(['Video_ID', 'Frame_s']):
    actual_label = frame_df['Ground_Truth_Label'].mode()[0]
    total_score = sum(row['Predicted_Joint_Label'] * weights.get(row['Joint_Name'], 0) for _, row in frame_df.iterrows())
    val_frame_records.append({'actual': actual_label, 'score': total_score})

df_val_eval = pd.DataFrame(val_frame_records)
y_val_true = df_val_eval['actual'].values

print("=== GRID SEARCH: TÌM NGƯỠNG ĐIỂM TỐI ƯU T* TRÊN TẬP VALIDATION ===")
print(f"{'Threshold (T)':<14} | {'Accuracy':<10} | {'Recall FAIL (0)':<16} | {'Recall PASS (1)':<16} | {'Macro-F1':<10}")
print("-" * 80)

val_results = []
for thresh in np.arange(6.0, 9.1, 0.1):
    y_pred_temp = (df_val_eval['score'].values >= thresh).astype(int)
    acc = accuracy_score(y_val_true, y_pred_temp)
    rec_0 = recall_score(y_val_true, y_pred_temp, pos_label=0, zero_division=0)
    rec_1 = recall_score(y_val_true, y_pred_temp, pos_label=1, zero_division=0)
    f1 = f1_score(y_val_true, y_pred_temp, average='macro', zero_division=0)
    
    val_results.append({'thresh': thresh, 'acc': acc, 'rec_0': rec_0, 'rec_1': rec_1, 'f1': f1})
    print(f"{thresh:<14.2f} | {acc*100:<9.2f}% | {rec_0*100:<15.2f}% | {rec_1*100:<15.2f}% | {f1:.4f}")

# Tự động tìm ngưỡng tối ưu T* theo tiêu chuẩn: Ưu tiên Recall FAIL >= 85% và Macro-F1 cao nhất
best_thresh = None
best_val_f1 = -1.0

for r in val_results:
    if r['rec_0'] >= 0.85 and r['f1'] > best_val_f1:
        best_val_f1 = r['f1']
        best_thresh = r['thresh']

# Fallback: nếu không thỏa mãn rec_0 >= 85%, chọn ngưỡng có Macro-F1 tuyệt đối cao nhất
if best_thresh is None:
    best_thresh = max(val_results, key=lambda r: r['f1'])['thresh']
    best_val_f1 = max(val_results, key=lambda r: r['f1'])['f1']

print("=" * 80)
print(f"-> NGƯỠNG TỐI ƯU ĐƯỢC CHỐT TỪ VALIDATION: T* = {best_thresh:.1f} (Validation Macro-F1 = {best_val_f1:.4f})")
print("   Ngưỡng này sẽ được CỐ ĐỊNH HOÀN TOÀN để đưa sang đánh giá tập Test độc lập.")
print("=" * 80)


=== GRID SEARCH: TÌM NGƯỠNG ĐIỂM TỐI ƯU T* TRÊN TẬP VALIDATION ===
Threshold (T)  | Accuracy   | Recall FAIL (0)  | Recall PASS (1)  | Macro-F1  
--------------------------------------------------------------------------------
6.00           | 68.61    % | 5.41           % | 100.00         % | 0.4562
6.10           | 69.73    % | 9.12           % | 99.83          % | 0.4909
6.20           | 69.73    % | 9.12           % | 99.83          % | 0.4909
6.30           | 69.73    % | 9.12           % | 99.83          % | 0.4909
6.40           | 69.73    % | 9.12           % | 99.83          % | 0.4909
6.50           | 69.73    % | 9.12           % | 99.83          % | 0.4909
6.60           | 69.73    % | 9.12           % | 99.83          % | 0.4909
6.70           | 69.73    % | 9.12           % | 99.83          % | 0.4909
6.80           | 69.73    % | 9.12           % | 99.83          % | 0.4909
6.90           | 69.73    % | 9.12           % | 99.83          % | 0.4909
7.00           | 69.73 

## 7. ĐÁNH GIÁ TRÊN TẬP TEST ĐỘC LẬP VỚI NGƯỠNG $T^*$ ĐÃ CỐ ĐỊNH

Áp dụng đúng mô hình CatBoost đã huấn luyện và ngưỡng $T^* = 8.1$ đã tối ưu từ tập Validation lên tập **Test độc lập** (`TEST_VIDEO_IDS = [5, 9]`).
Tại bước này, **tuyệt đối không quét lại hay điều chỉnh ngưỡng**, đảm bảo quy trình hoàn toàn khách quan và không rò rỉ dữ liệu.


In [9]:
# Dự đoán nhãn khớp trên tập TEST
test_data['Predicted_Joint_Label'] = cb_model.predict(X_test)

# Tính tổng điểm cấp độ khung hình trên tập TEST
test_frame_records = []
for (vid, fid), frame_df in test_data.groupby(['Video_ID', 'Frame_s']):
    actual_label = frame_df['Ground_Truth_Label'].mode()[0]
    total_score = sum(row['Predicted_Joint_Label'] * weights.get(row['Joint_Name'], 0) for _, row in frame_df.iterrows())
    test_frame_records.append({'actual': actual_label, 'score': total_score})

df_test_eval = pd.DataFrame(test_frame_records)
y_test_true = df_test_eval['actual'].values

# ÁP DỤNG NGƯỠNG T* ĐÃ CỐ ĐỊNH TỪ TẬP VALIDATION
final_test_pred = (df_test_eval['score'].values >= best_thresh).astype(int)

final_test_acc = accuracy_score(y_test_true, final_test_pred)
final_test_f1 = f1_score(y_test_true, final_test_pred, average='macro', zero_division=0)
rec_fail = recall_score(y_test_true, final_test_pred, pos_label=0, zero_division=0)
rec_pass = recall_score(y_test_true, final_test_pred, pos_label=1, zero_division=0)

print("==========================================================================")
print(f"      KẾT QUẢ ĐÁNH GIÁ CHUNG CUỘC TRÊN TẬP TEST (VIDEOS {TEST_VIDEO_IDS})")
print(f"                 ÁP DỤNG NGƯỠNG CỐ ĐỊNH T* = {best_thresh:.1f}")
print("==========================================================================\n")
print(classification_report(y_test_true, final_test_pred, target_names=['Fail (0)', 'Pass (1)']))
print(f"-> ĐỘ CHÍNH XÁC (Accuracy) : {final_test_acc * 100:.2f}%")
print(f"-> MACRO-F1               : {final_test_f1:.4f}")
print(f"-> ĐỘ NHẠY BẮT LỖI (Recall 0) : {rec_fail * 100:.2f}%")
print(f"-> ĐỘ CHÍNH XÁC ĐÚNG (Recall 1): {rec_pass * 100:.2f}%\n")

cm = confusion_matrix(y_test_true, final_test_pred)
if cm.shape == (2, 2):
    print(f"CHI TIẾT MA TRẬN NHẦM LẪN (CONFUSION MATRIX) TRÊN {len(y_test_true)} FRAMES TEST:")
    print(f"  [TN] Khung hình thực tế FAIL, máy chấm chuẩn FAIL : {cm[0][0]:<5} ({cm[0][0]/cm[0].sum()*100:.1f}%)")
    print(f"  [FP] Khung hình thực tế FAIL, máy bỏ lọt PASS    : {cm[0][1]:<5} ({cm[0][1]/cm[0].sum()*100:.1f}%)")
    print(f"  [FN] Khung hình thực tế PASS, máy phạt oan FAIL   : {cm[1][0]:<5} ({cm[1][0]/cm[1].sum()*100:.1f}%)")
    print(f"  [TP] Khung hình thực tế PASS, máy chấm chuẩn PASS : {cm[1][1]:<5} ({cm[1][1]/cm[1].sum()*100:.1f}%)")
print("==========================================================================")


      KẾT QUẢ ĐÁNH GIÁ CHUNG CUỘC TRÊN TẬP TEST (VIDEOS [5, 9])
                 ÁP DỤNG NGƯỠNG CỐ ĐỊNH T* = 8.1

              precision    recall  f1-score   support

    Fail (0)       0.84      0.90      0.87       203
    Pass (1)       0.95      0.91      0.93       410

    accuracy                           0.91       613
   macro avg       0.89      0.91      0.90       613
weighted avg       0.91      0.91      0.91       613

-> ĐỘ CHÍNH XÁC (Accuracy) : 90.86%
-> MACRO-F1               : 0.8988
-> ĐỘ NHẠY BẮT LỖI (Recall 0) : 90.15%
-> ĐỘ CHÍNH XÁC ĐÚNG (Recall 1): 91.22%

CHI TIẾT MA TRẬN NHẦM LẪN (CONFUSION MATRIX) TRÊN 613 FRAMES TEST:
  [TN] Khung hình thực tế FAIL, máy chấm chuẩn FAIL : 183   (90.1%)
  [FP] Khung hình thực tế FAIL, máy bỏ lọt PASS    : 20    (9.9%)
  [FN] Khung hình thực tế PASS, máy phạt oan FAIL   : 36    (8.8%)
  [TP] Khung hình thực tế PASS, máy chấm chuẩn PASS : 374   (91.2%)


## 8. KIỂM ĐỊNH MỞ RỘNG BẰNG STRATIFIED GROUP K-FOLD & SO SÁNH ĐỐI CHỨNG

### 8.1 Quy trình lọc nhiễu tĩnh nhất quán & Stratified Group 4-Fold CV
- **Loại bỏ nhiễu tĩnh tự động ở Phase 1:** Với `variance_threshold = 0.020`, Video 6 (`std = 0.0193`) được loại bỏ một cách khách quan ngay ở bước tiền xử lý thay vì giải thích ngoại lệ thủ công.
- **Khắc phục mất cân bằng nhãn giữa các Fold:** Sử dụng **`StratifiedGroupKFold(n_splits=4)`** trên 10 video hợp lệ trong Train Pool (`[1, 2, 3, 4, 7, 8, 10, 11, 12, 15]`) để đảm bảo:
  1. Không có khung hình nào của cùng 1 video bị rò rỉ giữa Train và Validation (*Group-aware*).
  2. Tỷ lệ PASS/FAIL ở mỗi Fold được phân bổ cân bằng nhất có thể (*Stratified*).
  3. Khởi tạo mới hoàn toàn `CatBoostClassifier` ở mỗi Fold và giữ nguyên logic tính $S_{\text{frame}}$ & Grid Search.


In [10]:
train_pool_df = master_dataset[~master_dataset['Video_ID'].isin(TEST_VIDEO_IDS)].copy().reset_index(drop=True)

# 1. Kiểm tra phân bố nhãn PASS/FAIL cấp khung hình của từng video trong Train Pool
print("=== PHÂN BỐ NHÃN KHUNG HÌNH TỪNG VIDEO TRONG TRAIN POOL ===")
for vid, vdf in train_pool_df.groupby('Video_ID'):
    f_labels = vdf.groupby('Frame_s')['Ground_Truth_Label'].apply(lambda s: s.mode()[0])
    n_pass = int((f_labels == 1).sum())
    n_fail = int((f_labels == 0).sum())
    tot = len(f_labels)
    print(f"  Video {int(vid):2d}: {tot:4d} frames | PASS = {n_pass:4d} ({n_pass/tot*100:5.1f}%) | FAIL = {n_fail:4d} ({n_fail/tot*100:5.1f}%)")

# 2. Chạy StratifiedGroupKFold (4-Fold cân bằng nhãn theo Video_ID)
sgkf = StratifiedGroupKFold(n_splits=4)
X_pool = train_pool_df[feature_cols]
y_pool = train_pool_df['Ground_Truth_Label']
groups_pool = train_pool_df['Video_ID']

thresholds = np.arange(6.0, 9.1, 0.1)
oof_preds = np.zeros(len(train_pool_df))

print("\n=== KIỂM CHỨNG ĐỘ ỔN ĐỊNH QUA STRATIFIED GROUP 4-FOLD CV ===")
print(f"{'Fold':<6} | {'Validation Videos':<18} | {'PASS / FAIL (Frames)':<22} | {'Best T*':<8} | {'Val Macro-F1':<12} | {'Val Acc':<10}")
print("-" * 88)

fold_t_list = []
fold_acc_list = []
fold_f1_list = []

for fold, (tr_idx, va_idx) in enumerate(sgkf.split(X_pool, y_pool, groups_pool)):
    tr_data = train_pool_df.iloc[tr_idx]
    va_data = train_pool_df.iloc[va_idx].copy()
    va_vids = sorted([int(v) for v in va_data['Video_ID'].unique()])
    
    # Khởi tạo mới hoàn toàn mô hình CatBoost cho từng fold (không dùng lại trọng số fold trước)
    cb_fold = CatBoostClassifier(iterations=150, depth=6, learning_rate=0.08, random_seed=SEED + fold, verbose=0)
    cb_fold.fit(tr_data[feature_cols], tr_data['Ground_Truth_Label'])
    
    pred_va = cb_fold.predict(va_data[feature_cols])
    va_data['pred_joint'] = pred_va
    oof_preds[va_idx] = pred_va
    
    # Tính điểm khung hình S_frame trên tập Validation của fold
    records = []
    for (vid, fid), fdf in va_data.groupby(['Video_ID', 'Frame_s']):
        actual = fdf['Ground_Truth_Label'].mode()[0]
        score = sum(row['pred_joint'] * weights.get(row['Joint_Name'], 0) for _, row in fdf.iterrows())
        records.append({'actual': actual, 'score': score})
    rdf = pd.DataFrame(records)
    y_true = rdf['actual'].values
    scores = rdf['score'].values
    
    n_p = int((y_true == 1).sum())
    n_f = int((y_true == 0).sum())
    
    # Grid Search ngưỡng T* trên fold (ưu tiên Recall FAIL >= 85% & Macro-F1 cao nhất)
    best_t, best_f1, best_acc = None, -1.0, 0.0
    for t in thresholds:
        yp = (scores >= t).astype(int)
        rec_0 = recall_score(y_true, yp, pos_label=0, zero_division=0)
        f1 = f1_score(y_true, yp, average='macro', zero_division=0)
        acc = accuracy_score(y_true, yp)
        if rec_0 >= 0.85 and f1 > best_f1:
            best_f1 = f1
            best_t = t
            best_acc = acc
    if best_t is None:
        best_t = max(thresholds, key=lambda t: f1_score(y_true, (scores >= t).astype(int), average='macro', zero_division=0))
        best_f1 = f1_score(y_true, (scores >= best_t).astype(int), average='macro', zero_division=0)
        best_acc = accuracy_score(y_true, (scores >= best_t).astype(int))
        
    fold_t_list.append(best_t)
    fold_acc_list.append(best_acc)
    fold_f1_list.append(best_f1)
    
    pf_str = f"{n_p:4d} / {n_f:4d} ({n_p/(n_p+n_f)*100:4.1f}% P)"
    print(f"Fold {fold+1:<1} | {str(va_vids):<18} | {pf_str:<22} | {best_t:<8.1f} | {best_f1:<12.4f} | {best_acc*100:<9.2f}%")

print("-" * 88)
print(f"-> Trung bình qua 4 Folds : Accuracy = {np.mean(fold_acc_list)*100:.2f}% | Macro-F1 = {np.mean(fold_f1_list):.4f}")
print(f"-> Ngưỡng T* trung bình   : {np.mean(fold_t_list):.2f} (Median: {np.median(fold_t_list):.1f})")


=== PHÂN BỐ NHÃN KHUNG HÌNH TỪNG VIDEO TRONG TRAIN POOL ===
  Video  1:  486 frames | PASS =  479 ( 98.6%) | FAIL =    7 (  1.4%)
  Video  2:  368 frames | PASS =  352 ( 95.7%) | FAIL =   16 (  4.3%)
  Video  3:  446 frames | PASS =  446 (100.0%) | FAIL =    0 (  0.0%)
  Video  4:  726 frames | PASS =  596 ( 82.1%) | FAIL =  130 ( 17.9%)
  Video  7:  446 frames | PASS =  150 ( 33.6%) | FAIL =  296 ( 66.4%)
  Video  8:  483 frames | PASS =  146 ( 30.2%) | FAIL =  337 ( 69.8%)
  Video 10: 1041 frames | PASS =   42 (  4.0%) | FAIL =  999 ( 96.0%)
  Video 11:  243 frames | PASS =   47 ( 19.3%) | FAIL =  196 ( 80.7%)
  Video 12:  755 frames | PASS =  755 (100.0%) | FAIL =    0 (  0.0%)
  Video 15:  442 frames | PASS =    0 (  0.0%) | FAIL =  442 (100.0%)

=== KIỂM CHỨNG ĐỘ ỔN ĐỊNH QUA STRATIFIED GROUP 4-FOLD CV ===
Fold   | Validation Videos  | PASS / FAIL (Frames)   | Best T*  | Val Macro-F1 | Val Acc   
--------------------------------------------------------------------------------------